# Solutions — Idempotent Pipelines: Deterministic IDs, Atomic Partitions, Run Ledgers

Exercise notebook: `08_data_engineering_python/06_idempotent_pipelines.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Deterministic run IDs

**Working solution**

In [ ]:
import hashlib
import json

def run_id(job_name, params):
    canonical = json.dumps(params, sort_keys=True, separators=(",", ":"), default=str)
    return f"{job_name}-{hashlib.sha256(canonical.encode()).hexdigest()[:12]}"

**Explanation**

Random UUIDs make every retry look like a new run. A content-derived id lets you detect 'this exact run already happened'. Canonical JSON is the trick for hashing dicts reliably.

**Complexity:** O(params).

**Common mistakes**

- Hashing `str(params)` — dict ordering/formatting differences change the hash.

## Exercise 2 — 🟡 Medium · Overwrite a partition atomically

**Working solution**

In [ ]:
import json
import shutil
import uuid
from pathlib import Path

def write_partition(root, partition, rows, filename="part-00000.jsonl"):
    target = Path(root).joinpath(*(f"{k}={v}" for k, v in partition.items()))
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = target.parent / f".tmp-{uuid.uuid4().hex}"
    staging.mkdir()
    try:
        with (staging / filename).open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, sort_keys=True, default=str) + "\n")
        if target.exists():
            shutil.rmtree(target)
        staging.rename(target)
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise
    return target

**Explanation**

'Insert overwrite partition' is the idempotent write: the partition's content is a pure function of its inputs. Staging + rename keeps the window where the partition is missing tiny
(on S3 you'd write to a new prefix and swap a pointer/manifest — Iceberg/Delta do exactly this).

**Complexity:** O(rows).

**Common mistakes**

- Appending to the partition on each run → duplicates on retry.
- Deleting the old partition before the new one is fully written.

## Exercise 3 — 🟡 Medium · Append with deduplication

**Working solution**

In [ ]:
import json
from pathlib import Path

def append_unique(path, records, key):
    path = Path(path)
    existing = set()
    if path.exists():
        with path.open(encoding="utf-8") as f:
            existing = {json.loads(line)[key] for line in f if line.strip()}
    path.parent.mkdir(parents=True, exist_ok=True)
    appended = 0
    with path.open("a", encoding="utf-8") as f:
        for record in records:
            if record[key] in existing:
                continue
            existing.add(record[key])
            f.write(json.dumps(record, sort_keys=True, default=str) + "\n")
            appended += 1
    return appended

**Explanation**

When you must append (event logs, audit trails), make the append *conditional* on a key. For big targets, keep the key set in an index/DB rather than re-reading the file.

**Complexity:** O(existing + new).

**Common mistakes**

- Appending blindly on retry.

## Exercise 4 — 🟠 Hard · A run ledger

**Working solution**

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

class RunLedger:
    def __init__(self, path, stale_after_seconds=3600, clock=lambda: datetime.now(timezone.utc)):
        self.path = Path(path)
        self.stale_after = timedelta(seconds=stale_after_seconds)
        self.clock = clock
        self._entries = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}

    @staticmethod
    def _key(job, partition):
        return f"{job}|{partition}"

    def _save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(self._entries, indent=2, sort_keys=True), encoding="utf-8")
        tmp.replace(self.path)

    def should_run(self, job, partition, input_hash, force=False):
        entry = self._entries.get(self._key(job, partition))
        if entry is None or force:
            return True
        if entry["status"] == "running":
            started = datetime.fromisoformat(entry["started_at"])
            return self.clock() - started >= self.stale_after
        return not (entry["status"] == "success" and entry["input_hash"] == input_hash)

    def start(self, job, partition, input_hash):
        key = self._key(job, partition)
        previous = self._entries.get(key, {})
        self._entries[key] = {"status": "running", "input_hash": input_hash, "started_at": self.clock().isoformat(),
                              "finished_at": None, "stats": None, "error": None, "attempts": previous.get("attempts", 0) + 1}
        self._save()

    def _finish(self, job, partition, **fields):
        entry = self._entries[self._key(job, partition)]
        entry.update(finished_at=self.clock().isoformat(), **fields)
        self._save()

    def succeed(self, job, partition, stats):
        self._finish(job, partition, status="success", stats=stats, error=None)

    def fail(self, job, partition, error):
        self._finish(job, partition, status="failed", error=str(error))

    def history(self):
        return json.loads(json.dumps(self._entries))

**Explanation**

The ledger answers "has this exact work already been done?" — keyed by *what* (job + partition) and *with which inputs* (hash). Staleness handles crashed runs that never wrote
`failed`. This is the control table you'd keep in DynamoDB for Glue jobs, or what Airflow's task-instance table does.

**Complexity:** O(entries) per save.

**Common mistakes**

- Treating 'running' forever as running (a crash wedges the partition).
- Skipping reruns even when the input changed.

## Challenge — 🔴 Very Hard · An idempotent daily job end-to-end

**Working solution**

In [ ]:
import csv
import hashlib
import json
import shutil
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

STATUSES = {"PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"}

def _transform(raw):
    qty = int(raw["quantity"])
    price = float(raw["unit_price"])
    if qty < 1 or price < 0 or not raw.get("order_id") or not raw.get("customer_id"):
        raise ValueError("invalid quantity/price/ids")
    status = raw["status"].strip().upper()
    if status not in STATUSES:
        raise ValueError(f"bad status {status}")
    ts = datetime.fromisoformat(raw["order_ts"].replace("Z", "+00:00"))
    upd = datetime.fromisoformat(raw["updated_at"].replace("Z", "+00:00"))
    return {"order_id": raw["order_id"], "customer_id": raw["customer_id"], "product_id": raw["product_id"], "quantity": qty,
            "unit_price": price, "amount": round(qty * price, 2), "status": status, "order_ts": ts.isoformat(), "updated_at": upd.isoformat(),
            "channel": raw["channel"].lower(), "country": raw["country"].upper()}

def _file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _input_hash(files):
    digest = hashlib.sha256()
    for f in sorted(files, key=lambda p: p.name):
        digest.update(f"{f.name}:{_file_hash(f)}\n".encode())
    return digest.hexdigest()

def _write_jsonl_atomic(target_dir, rows):
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    staging = target_dir.parent / f".tmp-{uuid.uuid4().hex}"
    staging.mkdir()
    with (staging / "part-00000.jsonl").open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, sort_keys=True) + "\n")
    if target_dir.exists():
        shutil.rmtree(target_dir)
    staging.rename(target_dir)

class _Ledger:
    def __init__(self, path):
        self.path = Path(path)
        self.entries = json.loads(self.path.read_text()) if self.path.exists() else {}
    def done(self, partition, h):
        e = self.entries.get(partition)
        return e is not None and e["status"] == "success" and e["input_hash"] == h
    def record(self, partition, h, status, stats=None, error=None):
        self.entries[partition] = {"status": status, "input_hash": h, "stats": stats, "error": error,
                                   "at": datetime.now(timezone.utc).isoformat()}
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(self.entries, indent=2, sort_keys=True))

def run_daily_job(daily_dir, output_root, ledger_path, force=False):
    ledger, out = _Ledger(ledger_path), Path(output_root)
    result = {"processed": [], "skipped": [], "failed": [], "stats": {}}
    for part_dir in sorted(p for p in Path(daily_dir).glob("dt=*") if p.is_dir()):
        date = part_dir.name.split("=", 1)[1]
        files = sorted(part_dir.glob("*.csv"))
        h = _input_hash(files)
        if not force and ledger.done(date, h):
            result["skipped"].append(date)
            continue
        try:
            stats = {"read": 0, "loaded": 0, "rejected": 0, "duplicates": 0}
            best, rejects = {}, []
            for f in files:
                with f.open(newline="", encoding="utf-8-sig") as fh:
                    for line_no, raw in enumerate(csv.DictReader(fh), start=2):
                        stats["read"] += 1
                        try:
                            rec = _transform(raw)
                        except (ValueError, KeyError, AttributeError, TypeError) as exc:
                            rejects.append({**raw, "_error": str(exc), "_source": f.name, "_line": line_no})
                            continue
                        prev = best.get(rec["order_id"])
                        if prev is not None:
                            stats["duplicates"] += 1
                            if prev["updated_at"] >= rec["updated_at"]:
                                continue
                        best[rec["order_id"]] = rec
            stats["rejected"], stats["loaded"] = len(rejects), len(best)
            _write_jsonl_atomic(out / "orders" / f"dt={date}", [best[k] for k in sorted(best)])
            _write_jsonl_atomic(out / "rejects" / f"dt={date}", rejects)
            ledger.record(date, h, "success", stats)
            result["processed"].append(date)
            result["stats"][date] = stats
        except Exception as exc:
            ledger.record(date, h, "failed", error=f"{type(exc).__name__}: {exc}")
            result["failed"].append(date)
    return result

**Explanation**

Everything from this section composes: content hashing detects change, the ledger makes reruns cheap and safe, per-partition atomic overwrite makes outputs a pure function of inputs,
dedup inside the partition removes within-day duplicates. Note the day-6 re-delivered file is *inside* a different partition — dedup across partitions needs a global key index
(that's Project 4).

**Complexity:** O(total input bytes) on the first run, O(hashing) on reruns.

**Common mistakes**

- Keying the ledger on file names only — a file corrected in place would never be reprocessed.
- Letting one bad partition abort the whole backfill.